# Exploratory Revision: Biodiversity Loss and UK Beef Imports

**Research question:** Is biodiversity loss in beef-exporting countries associated with the volume of beef exports to the United Kingdom?

This notebook integrates prior cleaning and panel analysis outputs using only existing pipeline data.

## 1) Setup and Data Load

No global warning suppression is used.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
sns.set_style('whitegrid')

DATA = os.path.dirname(os.path.abspath('__file__'))
CLEAN = os.path.join(DATA, 'clean')
FIGURES = os.path.join(DATA, 'figures')
SAVE_FIGURES = False
if SAVE_FIGURES:
    os.makedirs(FIGURES, exist_ok=True)

panel_raw = pd.read_csv(os.path.join(CLEAN, 'panel_2001_2014.csv'))
cs = pd.read_csv(os.path.join(CLEAN, 'crosssection_avg.csv'))
panel_ext = pd.read_csv(os.path.join(CLEAN, 'panel_extended_2010_2020.csv'))

print(f'panel_raw: {panel_raw.shape}, countries={panel_raw.iso3.nunique()}, years={panel_raw.year.min()}-{panel_raw.year.max()}')
print(f'crosssection_avg: {cs.shape}')
print(f'panel_extended_2010_2020: {panel_ext.shape}')


## 2) Variance Decomposition and Implication

If `bii_loss` variation is mostly between countries, FE has lower power because FE uses within-country movement only.

In [ ]:
def variance_decomp(df, col, id_col='iso3'):
    overall = df[col].var()
    between = df.groupby(id_col)[col].mean().var()
    within = df.groupby(id_col)[col].apply(lambda s: s - s.mean()).var()
    return pd.Series({'overall_var': overall, 'between_var': between, 'within_var': within, 'between_pct': 100*between/overall, 'within_pct': 100*within/overall})

vd = pd.DataFrame({'bii_loss': variance_decomp(panel_raw, 'bii_loss'), 'log_uk_import_qty_t': variance_decomp(panel_raw, 'log_uk_import_qty_t')}).T
display(vd.round(4))


## 3) Core Panel Models (Comparison-Oriented)

Pooled OLS is kept for comparison and the poolability test only.

In [ ]:
panel = panel_raw.set_index(['iso3', 'year']).sort_index()
y = panel['log_uk_import_qty_t']
x = panel[['bii_loss']]
x_const = sm.add_constant(x)

pooled_res = PooledOLS(y, x_const).fit(cov_type='clustered', cluster_entity=True)
fe_res = PanelOLS(y, x, entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
re_res = RandomEffects(y, x_const).fit(cov_type='clustered', cluster_entity=True)
fe2_res = PanelOLS(y, x, entity_effects=True, time_effects=True).fit(cov_type='clustered', cluster_entity=True)

print(f"Poolability F-test p-value: {fe_res.f_pooled.pval:.6f}")
b_fe, b_re = fe_res.params['bii_loss'], re_res.params['bii_loss']
v_fe, v_re = fe_res.std_errors['bii_loss']**2, re_res.std_errors['bii_loss']**2
if v_fe > v_re:
    hausman = (b_fe - b_re)**2 / (v_fe - v_re)
    p_haus = 1 - stats.chi2.cdf(hausman, 1)
else:
    p_haus = np.nan
preferred = 'FE (entity)' if (not np.isnan(p_haus) and p_haus < 0.05) else 'RE'
print(f'Preferred model: {preferred}')

comparison = pd.DataFrame([
    ['Pooled OLS (comparison only)', pooled_res.params['bii_loss'], pooled_res.std_errors['bii_loss'], pooled_res.pvalues['bii_loss'], int(pooled_res.nobs)],
    ['FE (entity) baseline', fe_res.params['bii_loss'], fe_res.std_errors['bii_loss'], fe_res.pvalues['bii_loss'], int(fe_res.nobs)],
    ['FE (entity + time) robustness', fe2_res.params['bii_loss'], fe2_res.std_errors['bii_loss'], fe2_res.pvalues['bii_loss'], int(fe2_res.nobs)],
    ['Random effects', re_res.params['bii_loss'], re_res.std_errors['bii_loss'], re_res.pvalues['bii_loss'], int(re_res.nobs)],
], columns=['model', 'beta_bii_loss', 'se', 'p_value', 'N'])
display(comparison.round(4))


In [ ]:
effects = fe_res.estimated_effects.reset_index().drop_duplicates('iso3')
effects = effects.rename(columns={'estimated_effects': 'alpha_i'})
effects = effects.merge(panel_raw[['iso3', 'country']].drop_duplicates(), on='iso3', how='left').sort_values('alpha_i')
print('Lowest fixed effects (5):')
display(effects[['iso3', 'country', 'alpha_i']].head(5).round(4))
print('Highest fixed effects (5):')
display(effects[['iso3', 'country', 'alpha_i']].tail(5).round(4))


## 4) Lagged Predictor Robustness

In [ ]:
lag_df = panel_raw.sort_values(['iso3', 'year']).copy()
lag_df['bii_loss_l1'] = lag_df.groupby('iso3')['bii_loss'].shift(1)
lag_df['bii_loss_l2'] = lag_df.groupby('iso3')['bii_loss'].shift(2)
lag_panel = lag_df.set_index(['iso3', 'year'])
fe_l1 = PanelOLS(lag_panel.dropna(subset=['bii_loss_l1'])['log_uk_import_qty_t'], lag_panel.dropna(subset=['bii_loss_l1'])[['bii_loss_l1']], entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
fe_l2 = PanelOLS(lag_panel.dropna(subset=['bii_loss_l2'])['log_uk_import_qty_t'], lag_panel.dropna(subset=['bii_loss_l2'])[['bii_loss_l2']], entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
display(pd.DataFrame([
    ['FE contemporaneous', fe_res.params['bii_loss'], fe_res.std_errors['bii_loss'], fe_res.pvalues['bii_loss']],
    ['FE lag1', fe_l1.params['bii_loss_l1'], fe_l1.std_errors['bii_loss_l1'], fe_l1.pvalues['bii_loss_l1']],
    ['FE lag2', fe_l2.params['bii_loss_l2'], fe_l2.std_errors['bii_loss_l2'], fe_l2.pvalues['bii_loss_l2']],
], columns=['model', 'beta', 'se', 'p_value']).round(4))


## 5) Mundlak (Correlated Random Effects)

In [ ]:
mundlak = panel.copy()
mundlak['bii_loss_mean'] = mundlak.groupby(level='iso3')['bii_loss'].transform('mean')
mundlak_res = RandomEffects(mundlak['log_uk_import_qty_t'], sm.add_constant(mundlak[['bii_loss', 'bii_loss_mean']])).fit(cov_type='clustered', cluster_entity=True)
display(pd.DataFrame({'coef': mundlak_res.params, 'se': mundlak_res.std_errors, 'p_value': mundlak_res.pvalues}).round(4))


## 6) Cross-Sectional Scatter + Cook's D

In [ ]:
y_cs = cs['log_uk_import_qty_t'].astype(float)
X_cs = sm.add_constant(cs[['bii_loss']].astype(float))
ols_cs = sm.OLS(y_cs, X_cs).fit()
cs['cooks_d'] = ols_cs.get_influence().cooks_distance[0]
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(cs['bii_loss'], cs['log_uk_import_qty_t'], alpha=0.8)
x_line = np.linspace(cs['bii_loss'].min(), cs['bii_loss'].max(), 100)
ax.plot(x_line, ols_cs.params['const'] + ols_cs.params['bii_loss'] * x_line, color='firebrick', lw=2, label='OLS fit')
for _, r in cs.iterrows():
    ax.text(r['bii_loss'], r['log_uk_import_qty_t'], r['iso3'], fontsize=8, alpha=0.75)
ax.set_xlabel('bii_loss')
ax.set_ylabel('log_uk_import_qty_t')
ax.legend()
plt.tight_layout()
plt.show()
display(cs[['iso3', 'country', 'cooks_d']].sort_values('cooks_d', ascending=False).head(10).round(4))


## 7) Panel Influence, Serial Correlation, and Trade Threshold Sensitivity

In [ ]:
panel_diag = panel_raw.sort_values(['iso3', 'year']).copy()
panel_diag['y_within'] = panel_diag['log_uk_import_qty_t'] - panel_diag.groupby('iso3')['log_uk_import_qty_t'].transform('mean')
panel_diag['x_within'] = panel_diag['bii_loss'] - panel_diag.groupby('iso3')['bii_loss'].transform('mean')
within_ols = sm.OLS(panel_diag['y_within'], sm.add_constant(panel_diag[['x_within']])).fit()
panel_diag['cooks_d_fe_within'] = within_ols.get_influence().cooks_distance[0]
display(panel_diag[['iso3', 'country', 'year', 'cooks_d_fe_within']].sort_values('cooks_d_fe_within', ascending=False).head(12).round(4))

try:
    from wildboottest.wildboottest import wildboottest
    print(wildboottest(within_ols, param='x_within', cluster=panel_diag['iso3'], B=9999))
except Exception as e:
    print(f'wildboottest unavailable or incompatible: {e}')

fe_resid = fe_res.resids.reset_index()
last_col = fe_resid.columns[-1]
fe_resid = fe_resid.rename(columns={last_col: 'e'}).sort_values(['iso3', 'year'])
fe_resid['e_l1'] = fe_resid.groupby('iso3')['e'].shift(1)
fe_resid['de'] = fe_resid['e'] - fe_resid['e_l1']
wd = fe_resid.dropna(subset=['de', 'e_l1']).copy()
wd_model = sm.OLS(wd['de'], wd[['e_l1']]).fit(cov_type='cluster', cov_kwds={'groups': wd['iso3']})
z = (wd_model.params['e_l1'] + 0.5) / wd_model.bse['e_l1']
p = 2 * (1 - stats.norm.cdf(abs(z)))
print(f'Wooldridge-style serial correlation test p-value: {p:.4f}')

totals = panel_raw.groupby('iso3')['uk_import_qty_t'].sum()
rows = []
for t in [10, 50, 100]:
    keep = totals[totals >= t].index
    dft = panel_raw[panel_raw['iso3'].isin(keep)].set_index(['iso3', 'year']).sort_index()
    res = PanelOLS(dft['log_uk_import_qty_t'], dft[['bii_loss']], entity_effects=True).fit(cov_type='clustered', cluster_entity=True)
    rows.append([t, dft.index.get_level_values('iso3').nunique(), int(res.nobs), res.params['bii_loss'], res.std_errors['bii_loss'], res.pvalues['bii_loss']])
display(pd.DataFrame(rows, columns=['min_total_trade_t', 'countries', 'N', 'beta_bii_loss', 'se', 'p_value']).round(4))


## 8) Interpretation and Limits

- The model estimates association, not causal effects.
- Directionality caveat: `bii_loss` is interpreted as a proxy for cumulative land-use intensity.
- Two-way FE is treated as robustness because slow-moving `bii_loss` can be absorbed by time effects.
- The panel is unbalanced; historical BIO ends in 2014; SSP2 values are modelled projections.
- SSP2 slice can include years beyond merged trade years; inner joins keep the overlap only.